# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahd799/flyrank-internship-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
--
I will prioritize pages that have strong observed search visibility and are in a potentially useful ranking range.

The baseline rule uses two observed signals:
- gsc_impressions: higher impressions indicate stronger observed search visibility.
- gsc_avg_position: lower average position indicates better observed ranking.

Reason code:
- HIGH_VISIBILITY_OPPORTUNITY: the page has high observed impressions and a useful average position, so it is a candidate for review and possible refresh.

Action:
- REVIEW_REFRESH

The rule is decision-support only. It does not claim that refreshing the page will improve performance.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df[["report_date", "client_hash_id", "content_hash_id",
    "gsc_impressions", "gsc_avg_position"]].head()

Rows: 9841378
Columns: 31


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,2.272727


### Signal 1: gsc_impressions

Verdict: CONFIRMED

Higher impression buckets show stronger observed search visibility. I will use impressions as the main visibility signal for the baseline.

In [15]:
df["impressions_bucket"] = pd.cut(
    df["gsc_impressions"],
    bins=[-1, 0, 5, 20, 100, np.inf],
    labels=["0", "1-5", "6-20", "21-100", "101+"]
)

print(df["impressions_bucket"].value_counts(sort=False))

impressions_bucket
0         6230317
1-5       1100192
6-20       891854
21-100     985532
101+       633483
Name: count, dtype: int64


### Signal 2: gsc_avg_position

Verdict: CONFIRMED

Pages with higher impression buckets show lower average position in the observed data. This supports using average position as a secondary signal when interpreting visibility.

In [17]:
position_check = (
    df.groupby("impressions_bucket", observed=True)["gsc_avg_position"]
      .agg(["count", "mean", "median"])
)

print(position_check)

                      count       mean    median
impressions_bucket                              
0                         0        NaN       NaN
1-5                 1100192  20.878008  8.000000
6-20                 891854  17.416488  9.333333
21-100               985532  11.808591  7.105263
101+                 633483  11.066558  5.712000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
## Rule implementation

I will use observed impressions to create the baseline score because it directly measures search visibility.

The score is:
2 * log(1 + gsc_impressions)

Pages with high visibility and an average position at or below 20 receive the reason code HIGH_VISIBILITY_OPPORTUNITY and the action REVIEW_REFRESH.

This is a baseline ranking rule, not a prediction of future performance.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

queue = df[
    ["report_date",
     "client_hash_id",
     "content_hash_id",
     "gsc_impressions",
     "gsc_avg_position"]
].copy()

queue["score"] = 2 * np.log1p(queue["gsc_impressions"])

queue["reason_code"] = np.where(
    (queue["gsc_impressions"] >= 101) &
    (queue["gsc_avg_position"] <= 20),
    "HIGH_VISIBILITY_OPPORTUNITY",
    "MONITOR"
)

queue["action"] = np.where(
    queue["reason_code"] == "HIGH_VISIBILITY_OPPORTUNITY",
    "REVIEW_REFRESH",
    "MONITOR"
)

queue = queue.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

print(queue.head(20))

   report_date           client_hash_id           content_hash_id  \
0   2026-03-28  client_23a62021009f63c4  content_44f34c0a90047651   
1   2026-03-29  client_e547b89c05043229  content_eadb33b5df496f4a   
2   2026-03-04  client_62f4a7e64f5e0096  content_34a70fea29d15f24   
3   2026-03-28  client_e547b89c05043229  content_eadb33b5df496f4a   
4   2026-03-04  client_62f4a7e64f5e0096  content_945d6ff91386c817   
5   2026-03-30  client_e547b89c05043229  content_eadb33b5df496f4a   
6   2026-03-27  client_e547b89c05043229  content_eadb33b5df496f4a   
7   2026-03-31  client_e547b89c05043229  content_eadb33b5df496f4a   
8   2026-03-24  client_e547b89c05043229  content_eadb33b5df496f4a   
9   2026-03-30  client_73cda7b4e4f265ea  content_fec55986a1868d62   
10  2026-03-27  client_23a62021009f63c4  content_44f34c0a90047651   
11  2026-03-29  client_23a62021009f63c4  content_44f34c0a90047651   
12  2026-03-22  client_e547b89c05043229  content_eadb33b5df496f4a   
13  2026-03-23  client_e547b89c050

In [19]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_cols = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_avg_position",
    "score",
    "reason_code",
    "action"
]

queue[output_cols].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:", "work/outputs/baseline_action_score.csv")
print("Rows written:", len(queue))

Saved: work/outputs/baseline_action_score.csv
Rows written: 9841378


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
--
The top 20 rows are ranked by the baseline score. For each recommendation, I will record the action, reason code, confidence note, and what could make the recommendation wrong.

These are observed-data recommendations, not guaranteed opportunities.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the top 20 ranked pages for manual review

# Build the manual-review table for the top 20 pages

top20 = queue.head(20).copy()

top20["confidence_note"] = (
    "Medium confidence: strong observed visibility, "
    "but the baseline does not measure content quality or actual refresh impact."
)

top20["what_would_make_it_wrong"] = (
    "The page may not have a meaningful refresh opportunity, "
    "or the observed visibility may not translate into an actionable content change."
)

review_cols = [
    "report_date",
    "content_hash_id",
    "gsc_impressions",
    "gsc_avg_position",
    "score",
    "reason_code",
    "action",
    "confidence_note",
    "what_would_make_it_wrong"
]

print(top20[review_cols].to_string(index=False))

report_date          content_hash_id  gsc_impressions  gsc_avg_position     score                 reason_code         action                                                                                                            confidence_note                                                                                                                what_would_make_it_wrong
 2026-03-28 content_44f34c0a90047651            40084          0.083350 21.197515 HIGH_VISIBILITY_OPPORTUNITY REVIEW_REFRESH Medium confidence: strong observed visibility, but the baseline does not measure content quality or actual refresh impact. The page may not have a meaningful refresh opportunity, or the observed visibility may not translate into an actionable content change.
 2026-03-29 content_eadb33b5df496f4a            39305          2.197507 21.158265 HIGH_VISIBILITY_OPPORTUNITY REVIEW_REFRESH Medium confidence: strong observed visibility, but the baseline does not measure content quality or actual 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
--

The weakest recommendations are reviewed separately to see whether the rule is producing obviously poor candidates.

The baseline uses only gsc_impressions and gsc_avg_position. It does not use future-window outcomes, labels, or product flags.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check the weakest recommendations in the ranked queue

weak_picks = queue[
    queue["reason_code"] == "MONITOR"
].sort_values(
    "score",
    ascending=True
)

print("Number of monitor rows:", len(weak_picks))

print(
    weak_picks[output_cols].head(10)
)

Number of monitor rows: 9334705
        report_date           client_hash_id           content_hash_id  \
5083226  2026-03-07  client_a80fca3f171ed1de  content_9a93cd9714fa6122   
5083245  2026-03-17  client_f623b01661d4bfe4  content_04cf17b789c4f326   
5083246  2026-03-17  client_f623b01661d4bfe4  content_17248092ae616a99   
5083247  2026-03-17  client_f623b01661d4bfe4  content_15e6b6d346a83fce   
5083248  2026-03-17  client_f623b01661d4bfe4  content_bd8a4646212bc48e   
5083217  2026-03-17  client_f623b01661d4bfe4  content_05352ea62cf12b3e   
5083218  2026-03-17  client_f623b01661d4bfe4  content_03b13ac9f5bdd748   
5083219  2026-03-17  client_f623b01661d4bfe4  content_40f026159f9eaf07   
5083220  2026-03-17  client_f623b01661d4bfe4  content_862f87597137a8eb   
5083221  2026-03-17  client_f623b01661d4bfe4  content_e0eec228b31d74d6   

         gsc_impressions  gsc_avg_position  score reason_code   action  
5083226                0               NaN    0.0     MONITOR  MONITOR  
5083245

In [22]:
# Confirm which columns were used to build the baseline score

used_features = [
    "gsc_impressions",
    "gsc_avg_position"
]

print("Features used by baseline:")
print(used_features)

print("\nNo future-window or outcome/label columns were used.")

Features used by baseline:
['gsc_impressions', 'gsc_avg_position']

No future-window or outcome/label columns were used.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.